# 📚 Chatbot nội bộ Vinamilk với LangChain + FAISS
Notebook này gồm 2 phần:
1. Tạo vector database từ PDF
2. Chatbot hỏi–đáp bằng mô hình LLaMA và truy vấn vector

In [1]:
!pip install -r setup.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'setup.txt'


In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_community.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

# Khai báo biến
pdf_data_path = "../data"
vector_db_path = "vectorstores/db_faiss"

def create_db_from_files():
    # Khai bao loader de quet toan bo thu muc data
    loader = DirectoryLoader(pdf_data_path, glob="*.pdf", loader_cls = PyPDFLoader)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=256, chunk_overlap=20)
    chunks = text_splitter.split_documents(documents)

    # Embedding
    embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    db = FAISS.from_documents(chunks, embedding_model)
    db.save_local(vector_db_path)
    return db

create_db_from_files()

ModuleNotFoundError: No module named 'langchain_community'

---

In [ ]:
from langchain_community.llms import CTransformers
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Cau hinh
model_file = "../models/vinallama-7b-chat_q5_0.gguf"
vector_db_path = "vectorstores/db_faiss"

#Load LLM
def load_llm(model_file):
    llm = CTransformers(
        model=model_file,
        model_type="llama",
        max_new_tokens=1024,
        temperature=0.01)
    return llm

# Tao prompt template
def create_prompt(template):
    prompt = PromptTemplate(template=template, input_variables=["context","question"])
    return prompt

# Tao qa chain
def create_qa_chain(prompt, llm, db):
    llm_chain = RetrievalQA.from_chain_type(
        llm = llm,
        chain_type = "stuff",
        retriever = db.as_retriever(search_kwargs={"k":2}),
        return_source_documents = False,
        chain_type_kwargs={"prompt":prompt})
    return llm_chain

# Read tu Vector DB
def read_vectors_db():
    #Embedding
    embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    db = FAISS.load_local(vector_db_path, embedding_model, allow_dangerous_deserialization=True)
    return db

# Bat dau thu nghiem
db = read_vectors_db()
llm = load_llm(model_file)

#Tao Prompt
template = """Bạn là một trợ lý AI. Dưới đây là một số thông tin nội bộ của công ty:

{context}

Dựa vào thông tin trên, hãy trả lời câu hỏi sau:
Câu hỏi: {question}
Trả lời:"""

prompt = create_prompt(template)

llm_chain = create_qa_chain(prompt, llm, db)

In [ ]:
import sys
import contextlib
import io

In [ ]:
question = "Chế độ ưu đãi cho nhân viên của Vinamilk là gì?"

print("Đang tìm kiếm thông tin bạn cần...")

# Ẩn stdout tạm thời
with contextlib.redirect_stdout(io.StringIO()):
    response = llm_chain.invoke({"query": question})

print(f"Câu hỏi: {question}")
print("Kết quả:")
print(response["result"])


In [ ]:
question = "có bao nhiêu trang trại bò sữa?"

print("Đang tìm kiếm thông tin bạn cần...")

# Ẩn stdout tạm thời
with contextlib.redirect_stdout(io.StringIO()):
    response = llm_chain.invoke({"query": question})

print(f"Câu hỏi: {question}")
print("Kết quả:")
print(response["result"])

In [ ]:
question = "ai là Tổng giám đốc Vinamilk ?"

print("Đang tìm kiếm thông tin bạn cần...")

# Ẩn stdout tạm thời
with contextlib.redirect_stdout(io.StringIO()):
    response = llm_chain.invoke({"query": question})

print(f"Câu hỏi: {question}")
print("Kết quả:")
print(response["result"])

In [ ]:
question = "Mục tiêu của chính sách “Trẻ hóa đội ngũ quản lý” là gì?"

print("Đang tìm kiếm thông tin bạn cần...")

# Ẩn stdout tạm thời
with contextlib.redirect_stdout(io.StringIO()):
    response = llm_chain.invoke({"query": question})

print(f"Câu hỏi: {question}")
print("Kết quả:")
print(response["result"])

In [ ]:
import sys
import contextlib
import io

# Hàm chat tương tác
def chat():
    print('🤖 VinamilkBot: Xin chào! Bạn có thể hỏi tôi bất kỳ điều gì liên quan đến Vinamilk.')
    print("Gõ 'quit' để thoát.")
    while True:
        inp = input('Bạn: ')
        if inp.lower() == 'quit':
            print('🤖 VinamilkBot: Hẹn gặp lại bạn!')
            break

        print('Đang tìm kiếm thông tin bạn cần...')
        # Ẩn stdout tạm thời để tránh in log thừa
        with contextlib.redirect_stdout(io.StringIO()):
            response = llm_chain.invoke({"query": inp})

        print(f'🤖 VinamilkBot: {response["result"]}')

# Chạy chatbot
if __name__ == "__main__":
    chat()